# 相对估值分析（参数法）

## 分析目标
通过PE、PS、PB等相对估值指标，与行业内其他公司进行对比，分析估值回归情景。

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# 配置中文字体
from utils.direct_font_config import setup
setup()

from utils.font_manager import get_font_prop
font_prop = get_font_prop()

import tushare as ts

%matplotlib inline
sns.set_style('whitegrid')

print('✅ 相对估值分析模块加载成功')

✅ 使用字体: Heiti TC
   路径: /System/Library/Fonts/STHeiti Medium.ttc
✅ 使用系统字体: /System/Library/Fonts/STHeiti Medium.ttc
✅ 相对估值分析模块加载成功


## 1. 参数设置与数据获取

In [2]:
# ============================================================
# 配置参数
# ============================================================
STOCK_CODE = '300735.SZ'  # 光弘科技
STOCK_NAME = '光弘科技'
INDUSTRY = '电子'  # 行业分类

# Tushare Token（从环境变量获取）
import os
TS_TOKEN = os.environ.get('TUSHARE_TOKEN', '')

print(f'分析股票: {STOCK_CODE} - {STOCK_NAME}')
print(f'所属行业: {INDUSTRY}')
print(f'Tushare Token: {"已配置" if TS_TOKEN else "⚠️ 未设置环境变量 TUSHARE_TOKEN"}')

分析股票: 300735.SZ - 光弘科技
所属行业: 电子
Tushare Token: 已配置


## 2. 加载定增配置数据

In [3]:
from utils.config_loader import load_placement_config, print_config_summary

# 加载配置（指定数据目录）
project_params, risk_params, market_data = load_placement_config(STOCK_CODE, data_dir='../data')

# 打印配置摘要
print_config_summary(project_params, risk_params, market_data)

# 提取关键数据
current_price = project_params['current_price']
issue_price = project_params['issue_price']
net_income = project_params.get('net_income', 253532329.85)
net_assets = project_params.get('net_assets', 4939639031.34)
issue_shares = project_params['issue_shares']  # 定增发行股数

# 获取真实总股本和估值指标（从 Tushare daily_basic 接口）
import os
ts_token = os.environ.get('TUSHARE_TOKEN', '')
daily_basic_data = None  # 用于存储 daily_basic 数据，供后续使用

if ts_token:
    try:
        pro = ts.pro_api(ts_token)
        # 使用 daily_basic 接口获取总股本和估值指标（最新交易日）
        import datetime
        trade_date = (datetime.datetime.now() - datetime.timedelta(days=1)).strftime('%Y%m%d')
        df_basic = pro.daily_basic(ts_code=STOCK_CODE, trade_date=trade_date, 
                                   fields='ts_code,trade_date,total_share,float_share,total_mv,circ_mv,pe_ttm,pb,ps_ttm')
        
        if df_basic.empty:
            # 尝试获取最近的交易日数据（不指定日期）
            df_basic = pro.daily_basic(ts_code=STOCK_CODE,
                                       fields='ts_code,trade_date,total_share,float_share,pe_ttm,pb,ps_ttm')
        
        if not df_basic.empty:
            row = df_basic.iloc[0]
            total_shares = float(row['total_share']) * 10000  # 转换为股数（单位：万股）
            daily_basic_data = row  # 保存数据供后续使用
            print(f"\n✅ 从 Tushare 获取数据:")
            print(f"   总股本: {total_shares:,.0f} 股")
            print(f"   数据日期: {row['trade_date']}")
            print(f"   PE(TTM): {row['pe_ttm']:.2f} 倍")
            print(f"   PB: {row['pb']:.2f} 倍")
            print(f"   PS(TTM): {row['ps_ttm']:.2f} 倍")
        else:
            raise ValueError("daily_basic 未获取到数据")
    except Exception as e:
        print(f"\n⚠️ Tushare 获取数据失败: {e}")
        print("⚠️ 使用估算值")
        total_shares = 1000000000  # 估算值
else:
    print(f"\n⚠️ TUSHARE_TOKEN 环境变量未设置")
    print("⚠️ 使用估算值")
    total_shares = 1000000000  # 估算值

print(f"   定增发行股数: {issue_shares:,} 股")
print(f"   定增占比: {issue_shares/total_shares*100:.2f}%")

✅ 已加载定增参数: ../data/300735_SZ_placement_params.json
✅ 已加载市场数据: ../data/300735_SZ_market_data.json
   股票: 光弘科技 (300735.SZ)
   分析日期: 20260306
   当前价格: 23.88 元
✅ 使用市场数据中的最新价格: 23.88 元
✅ 使用真实市场数据:
   波动率: 30.63% (60日)
   收益率: -18.75% (60日年化)

📊 定增分析配置

📋 项目参数:
   发行价格: 20.25 元/股
   当前价格: 23.88 元/股
   锁定期: 6 个月
   发行数量: 5,000,000 股
   融资金额: 1.01 亿元
   当前收益率: +17.95% （浮盈）

📌 发行类型判断:
   MA30: 25.31 元
   发行价: 20.25 元
   ✅ 折价发行（有安全边际）
   安全边际: 20.01%

⚠️ 风险参数:
   波动率: 30.63%
   收益率(漂移率): -18.75%
   数据来源: market_data

📈 波动率详情:
   30日: 33.96%
   60日: 30.63%
   120日: 37.13%
   180日: 36.60%

✅ 从 Tushare 获取数据:
   总股本: 767,460,689 股
   数据日期: 20260306
   PE(TTM): 56.24 倍
   PB: 3.71 倍
   PS(TTM): 2.30 倍
   定增发行股数: 5,000,000 股
   定增占比: 0.65%


## 3. 计算当前公司的估值指标

In [4]:
# 使用 cell-5 中已获取的 TTM 估值指标
if daily_basic_data is not None:
    current_metrics = {
        'pe': daily_basic_data['pe_ttm'],
        'pb': daily_basic_data['pb'],
        'ps': daily_basic_data['ps_ttm']
    }
    print('\n=== 光弘科技估值指标（TTM）===')
    print(f"数据日期: {daily_basic_data['trade_date']}")
    print(f"当前股价: {current_price:.2f} 元")
    print(f"总股本: {total_shares:,.0f} 股")
    print(f"\n估值倍数（TTM）:")
    print(f"  PE (市盈率TTM): {current_metrics['pe']:.2f} 倍")
    print(f"  PB (市净率): {current_metrics['pb']:.2f} 倍")
    print(f"  PS (市销率TTM): {current_metrics['ps']:.2f} 倍")
else:
    # 降级方案：手动计算
    revenue = net_income / 0.15  # 假设净利率15%
    current_metrics = {
        'pe': current_price / (net_income / total_shares),
        'pb': current_price / (net_assets / total_shares),
        'ps': current_price / (revenue / total_shares)
    }
    print('\n=== 光弘科技估值指标（手动计算）===')
    print(f"当前股价: {current_price:.2f} 元")
    print(f"总股本: {total_shares:,.0f} 股")
    print(f"\n估值倍数（估算）:")
    print(f"  PE: {current_metrics['pe']:.2f} 倍")
    print(f"  PB: {current_metrics['pb']:.2f} 倍")
    print(f"  PS: {current_metrics['ps']:.2f} 倍")


=== 光弘科技估值指标（TTM）===
数据日期: 20260306
当前股价: 23.88 元
总股本: 767,460,689 股

估值倍数（TTM）:
  PE (市盈率TTM): 56.24 倍
  PB (市净率): 3.71 倍
  PS (市销率TTM): 2.30 倍


In [5]:
df_industry = pro.index_member_all(ts_code=STOCK_CODE)

In [6]:
df_industry

,l1_code,l1_name,l2_code,l2_name,l3_code,l3_name,ts_code,name,in_date,out_date,is_new
0,801080.SI,电子,801085.SI,消费电子,850854.SI,消费电子零部件及组装,300735.SZ,光弘科技,20171220,None,Y


In [7]:
df_industry = df_industry.sort_values('in_date', ascending=False)
latest_industry = df_industry.iloc[0]
latest_industry

l1_code      801080.SI
l1_name             电子
l2_code      801085.SI
l2_name           消费电子
l3_code      850854.SI
l3_name     消费电子零部件及组装
ts_code      300735.SZ
name              光弘科技
in_date       20171220
out_date          None
is_new               Y
Name: 0, dtype: object

## 4. 获取同行公司列表（基于行业分类）

In [8]:
# 获取同行公司数据（基于申万三级行业分类）
if TS_TOKEN:
    pro = ts.pro_api(TS_TOKEN)
    
    print('\n正在获取同行公司数据...')
    
    try:
        # 步骤1: 使用 index_member_all 接口获取股票的申万三级行业分类
        df_industry = pro.index_member_all(ts_code=STOCK_CODE)
        
        if df_industry.empty:
            raise ValueError(f"未获取到 {STOCK_CODE} 的行业分类信息")
        
        # 获取最新的行业分类信息
        df_industry = df_industry.sort_values('in_date', ascending=False)
        latest_industry = df_industry.iloc[0]
        
        target_index_code = latest_industry['l3_code']  # 申万三级行业指数代码
        target_industry_l3 = latest_industry.get('l3_name', '未知')  # 行业名称
        target_industry_l1 = latest_industry.get('l1_name', '未知')  # 一级行业
        target_name = latest_industry.get('name', '未知')  # 股票名称
        
        print(f"\n✅ 目标公司行业分类（申万三级）:")
        print(f"   公司名称: {target_name}")
        print(f"   申万三级行业: {target_industry_l3}")
        print(f"   申万一级行业: {target_industry_l1}")
        print(f"   行业指数代码: {target_index_code}")
        
        # 步骤2: 获取该申万三级行业的所有成分股
        print(f"\n正在获取 {target_industry_l3} 行业的成分股...")
        df_peers = pro.index_member_all(l3_code=target_index_code)
        # print(df_peers)
        
        if df_peers.empty:
            raise ValueError(f"未获取到行业 {target_index_code} 的成分股")
        
        # 排除目标公司
        df_peers = df_peers[df_peers['ts_code'] != STOCK_CODE]
        
        # 获取股票基本信息（市场等）
        peer_codes = df_peers['ts_code'].unique().tolist()
        peer_basic = pro.stock_basic(ts_code=','.join(peer_codes[:50]),  # 限制数量
                                     fields='ts_code,name,market')
        
        # 合并数据
        peer_stocks_all = pd.merge(df_peers, peer_basic, on='ts_code', how='left')
        peer_stocks_all = peer_stocks_all.drop_duplicates(subset=['ts_code'])
        # 优先选择相同市场的股票
        target_basic = pro.stock_basic(ts_code=STOCK_CODE, fields='ts_code,market')
        target_market = target_basic['market'].values[0] if not target_basic.empty else ''
        
        if target_market:
            same_market = peer_stocks_all[peer_stocks_all['market'] == target_market]
            other_market = peer_stocks_all[peer_stocks_all['market'] != target_market]
            peer_stocks_all = pd.concat([same_market, other_market])
        # print(peer_stocks_all)
        
        # 限制同行公司数量
        max_peers = 15
        if len(peer_stocks_all) > max_peers:
            peer_stocks_all = peer_stocks_all.head(max_peers)
        
        peer_codes = peer_stocks_all['ts_code'].tolist()
        peer_names_dict = dict(zip(peer_stocks_all['ts_code'], peer_stocks_all['name_x']))
        
        print(f"\n✅ 找到 {len(peer_codes)} 家同行公司")
        print(peer_codes)
        print(f"   市场: {target_market}")
        print(f"   同行列表: {list(peer_stocks_all['name_x'].values)}")
        
    except Exception as e:
        print(f"\n⚠️ 获取申万三级分类失败: {e}")
        print("⚠️ 提示: index_member_all 接口需要 2000 积分")
        print("⚠️ 降级使用申万一级分类")
        
        # 降级方案：使用申万一级分类
        try:
            basic_info = pro.stock_basic(ts_code=STOCK_CODE, 
                                         fields='ts_code,name,industry,market')
            target_industry_l1 = basic_info['industry'].values[0]
            target_name = basic_info['name'].values[0]
            target_market = basic_info['market'].values[0]
            
            print(f"\n✅ 目标公司（申万一级）:")
            print(f"   公司名称: {target_name}")
            print(f"   申万一级行业: {target_industry_l1}")
            
            # 获取同行业所有股票
            industry_stocks = pro.stock_basic(fields='ts_code,name,industry,market')
            peer_stocks_all = industry_stocks[
                (industry_stocks['industry'] == target_industry_l1) & 
                (industry_stocks['ts_code'] != STOCK_CODE)
            ].copy()
            
            # 优先同市场
            same_market = peer_stocks_all[peer_stocks_all['market'] == target_market]
            other_market = peer_stocks_all[peer_stocks_all['market'] != target_market]
            peer_stocks_all = pd.concat([same_market, other_market]).head(max_peers)
            
            peer_codes = peer_stocks_all['ts_code'].tolist()
            peer_names_dict = dict(zip(peer_stocks_all['ts_code'], peer_stocks_all['name']))
            
            print(f"\n✅ 找到 {len(peer_codes)} 家同行公司")
            print(f"   同行列表（前8家）: {list(peer_stocks_all.head(8)['name'].values)}")
            
        except Exception as e2:
            print(f"\n⚠️ 申万一级分类也失败: {e2}")
            print("⚠️ 使用手动设置的同行公司列表")
            peer_codes = ['002475.SZ', '002241.SZ', '002438.SZ', '300115.SZ']
            peer_names_dict = {
                '002475.SZ': '立讯精密',
                '002241.SZ': '歌尔股份',
                '002438.SZ': '蓝思科技',
                '300115.SZ': '长盈精密'
            }
else:
    print('\n⚠️ TUSHARE_TOKEN 环境变量未设置')
    print('⚠️ 使用示例数据进行分析')
    peer_codes = []


正在获取同行公司数据...

✅ 目标公司行业分类（申万三级）:
   公司名称: 光弘科技
   申万三级行业: 消费电子零部件及组装
   申万一级行业: 电子
   行业指数代码: 850854.SI

正在获取 消费电子零部件及组装 行业的成分股...

✅ 找到 15 家同行公司
['300136.SZ', '300115.SZ', '300322.SZ', '300433.SZ', '301123.SZ', '300647.SZ', '300686.SZ', '300679.SZ', '300709.SZ', '300787.SZ', '300793.SZ', '300916.SZ', '301285.SZ', '300857.SZ', '301491.SZ']
   市场: 创业板
   同行列表: ['信维通信', '长盈精密', '硕贝德', '蓝思科技', '奕东电子', '超频三', '智动力', '电连技术', '精研科技', '海能实业', '佳禾智能', '朗特智能', '鸿日达', '协创数据', '汉桑科技']


## 5. 获取同行公司估值指标（来自Tushare）

In [ ]:
import pandas as pd
import time
import datetime

peer_companies_data = None

if TS_TOKEN and peer_codes:
    print('\n正在获取同行公司估值指标...')
    
    try:
        pro = ts.pro_api(TS_TOKEN)
        peer_data_list = []
        
        # 获取最新交易日（避免拉取所有历史数据）
        trade_date = (datetime.datetime.now() - datetime.timedelta(days=1)).strftime('%Y%m%d')
        print(f"   获取日期: {trade_date}")
        
        # 逐个获取同行公司的 daily_basic 数据
        for idx, code in enumerate(peer_codes, 1):
            try:
                # 获取指定交易日的数据（只返回一天的数据）
                df_part = pro.daily_basic(
                    ts_code=code,
                    trade_date=trade_date,
                    fields='ts_code,trade_date,close,pe_ttm,pb,ps_ttm,total_mv'
                )
                
                if not df_part.empty:
                    # 添加公司名称
                    df_part['name'] = peer_names_dict.get(code, code)
                    peer_data_list.append(df_part)
                    print(f"   [{idx}/{len(peer_codes)}] {code}: OK")
                else:
                    print(f"   [{idx}/{len(peer_codes)}] {code}: 无数据")
                    
            except Exception as e:
                print(f"   [{idx}/{len(peer_codes)}] {code}: 失败 - {e}")
            
            # 避免请求过快（每次请求后暂停）
            time.sleep(0.3)
        
        if peer_data_list:
            peer_companies_data = pd.concat(peer_data_list, ignore_index=True)
            
            # 过滤掉 PE 或 PB 为空或异常的数据
            peer_companies_data = peer_companies_data[
                (peer_companies_data['pe_ttm'] > 0) & 
                (peer_companies_data['pe_ttm'] < 500) &
                (peer_companies_data['pb'] > 0) & 
                (peer_companies_data['pb'] < 20)
            ]
            
            # 重命名列：添加 ts_code -> code 的映射
            peer_companies_data = peer_companies_data.rename(columns={
                'ts_code': 'code',
                'pe_ttm': 'pe',
                'ps_ttm': 'ps',
                'total_mv': 'market_cap'
            })
            
            # 只保留需要的列
            peer_companies_data = peer_companies_data[['name', 'code', 'pe', 'ps', 'pb', 'market_cap']]
            peer_companies_data['code'] = peer_companies_data['code'].fillna('')
            
            print(f'\n✅ 成功获取 {len(peer_companies_data)} 家同行公司估值数据')
            print('\n=== 同行公司估值数据（来自Tushare）===')
            print(peer_companies_data.sort_values('market_cap', ascending=False).to_string(index=False))
        else:
            raise ValueError("未获取到同行公司数据")
            
    except Exception as e:
        print(f"\n⚠️ 获取同行估值数据失败: {e}")
        import traceback
        traceback.print_exc()
        print("⚠️ 使用示例数据")
        peer_companies_data = None

# 如果没有获取到数据，使用示例数据
if peer_companies_data is None or peer_companies_data.empty:
    print('\n=== 同行公司估值数据（示例）===')
    peer_companies_data = pd.DataFrame({
        'name': ['立讯精密', '歌尔股份', '蓝思科技', '长盈精密', '领益智造', '安洁科技', '比亚迪电子'],
        'code': ['002475.SZ', '002241.SZ', '002438.SZ', '300115.SZ', '002600.SZ', '002635.SZ', '002859.SZ'],
        'pe': [20.5, 25.8, 22.3, 28.6, 18.9, 32.1, 24.5],
        'ps': [1.2, 1.8, 1.5, 2.1, 1.0, 2.5, 1.6],
        'pb': [2.8, 3.2, 2.5, 3.8, 2.1, 4.2, 3.0],
        'market_cap': [120, 80, 50, 45, 65, 25, 90]
    })
    print(peer_companies_data.to_string(index=False))

# 计算行业平均值
industry_avg = {
    'pe': peer_companies_data['pe'].mean(),
    'ps': peer_companies_data['ps'].mean(),
    'pb': peer_companies_data['pb'].mean()
}

print('\n=== 行业平均估值 ===')
print(f"行业平均PE: {industry_avg['pe']:.2f} 倍")
print(f"行业平均PS: {industry_avg['ps']:.2f} 倍")
print(f"行业平均PB: {industry_avg['pb']:.2f} 倍")

# 计算当前公司与行业平均的偏离度
print('\n=== 估值偏离度分析 ===')
pe_deviation = (current_metrics['pe'] - industry_avg['pe']) / industry_avg['pe'] * 100
pb_deviation = (current_metrics['pb'] - industry_avg['pb']) / industry_avg['pb'] * 100
ps_deviation = (current_metrics['ps'] - industry_avg['ps']) / industry_avg['ps'] * 100

print(f"PE偏离度: {pe_deviation:+.1f}%")
print(f"PB偏离度: {pb_deviation:+.1f}%")
print(f"PS偏离度: {ps_deviation:+.1f}%")

## 6. 可视化对比分析

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 估值指标对比柱状图
ax1 = axes[0, 0]
metrics = ['PE', 'PS', 'PB']
current_vals = [current_metrics['pe'], current_metrics['ps'], current_metrics['pb']]
industry_vals = [industry_avg['pe'], industry_avg['ps'], industry_avg['pb']]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax1.bar(x - width/2, current_vals, width, label='光弘科技', color='#3498db', alpha=0.8)
bars2 = ax1.bar(x + width/2, industry_vals, width, label='行业平均', color='#e74c3c', alpha=0.8)

ax1.set_xlabel('估值指标', fontproperties=font_prop, fontsize=12)
ax1.set_ylabel('倍数', fontproperties=font_prop, fontsize=12)
ax1.set_title('相对估值对比', fontproperties=font_prop, fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, fontproperties=font_prop)
ax1.legend(prop=font_prop)
ax1.grid(True, alpha=0.3, axis='y')
for label in ax1.get_yticklabels():
    label.set_fontproperties(font_prop)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=10, fontproperties=font_prop)

# 2. PE对比
ax2 = axes[0, 1]
peer_names = list(peer_companies_data['name'])
peer_pes = list(peer_companies_data['pe'])
peer_pes.append(industry_avg['pe'])
peer_names.append('行业平均')
peer_pes.append(current_metrics['pe'])
peer_names.append('光弘科技')

colors_pe = ['#95a5a6'] * len(peer_companies_data) + ['#e74c3c', '#3498db']
bars = ax2.bar(range(len(peer_names)), peer_pes, color=colors_pe, alpha=0.7)
ax2.axhline(y=industry_avg['pe'], color='red', linestyle='--', alpha=0.5, label='行业平均')
ax2.set_xlabel('公司', fontproperties=font_prop, fontsize=10)
ax2.set_ylabel('PE (倍)', fontproperties=font_prop, fontsize=10)
ax2.set_title('PE倍数对比', fontproperties=font_prop, fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(peer_names)))
ax2.set_xticklabels(peer_names, fontproperties=font_prop, fontsize=9, rotation=45)
ax2.grid(True, alpha=0.3, axis='y')
for label in ax2.get_yticklabels():
    label.set_fontproperties(font_prop)

# 3. PS对比
ax3 = axes[1, 0]
peer_pss = list(peer_companies_data['ps'])
peer_pss.append(industry_avg['ps'])
peer_pss.append(current_metrics['ps'])

bars = ax3.bar(range(len(peer_names)), peer_pss, color=colors_pe, alpha=0.7)
ax3.axhline(y=industry_avg['ps'], color='red', linestyle='--', alpha=0.5)
ax3.set_xlabel('公司', fontproperties=font_prop, fontsize=10)
ax3.set_ylabel('PS (倍)', fontproperties=font_prop, fontsize=10)
ax3.set_title('PS倍数对比', fontproperties=font_prop, fontsize=12, fontweight='bold')
ax3.set_xticks(range(len(peer_names)))
ax3.set_xticklabels(peer_names, fontproperties=font_prop, fontsize=9, rotation=45)
ax3.grid(True, alpha=0.3, axis='y')
for label in ax3.get_yticklabels():
    label.set_fontproperties(font_prop)

# 4. PB对比
ax4 = axes[1, 1]
peer_pbs = list(peer_companies_data['pb'])
peer_pbs.append(industry_avg['pb'])
peer_pbs.append(current_metrics['pb'])

bars = ax4.bar(range(len(peer_names)), peer_pbs, color=colors_pe, alpha=0.7)
ax4.axhline(y=industry_avg['pb'], color='red', linestyle='--', alpha=0.5)
ax4.set_xlabel('公司', fontproperties=font_prop, fontsize=10)
ax4.set_ylabel('PB (倍)', fontproperties=font_prop, fontsize=10)
ax4.set_title('PB倍数对比', fontproperties=font_prop, fontsize=12, fontweight='bold')
ax4.set_xticks(range(len(peer_names)))
ax4.set_xticklabels(peer_names, fontproperties=font_prop, fontsize=9, rotation=45)
ax4.grid(True, alpha=0.3, axis='y')
for label in ax4.get_yticklabels():
    label.set_fontproperties(font_prop)

plt.suptitle('相对估值分析：与同行对比', fontproperties=font_prop, fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## 7. 估值回归情景分析

In [ ]:
# 分析：如果估值回归到行业平均水平，股价会如何变化？

print('\n=== 估值回归情景分析 ===')
print('\n情景1: 估值回归到行业平均水平')
print('-' * 50)

# 基于行业平均PE的目标价格
eps = net_income / total_shares  # 每股收益
target_price_pe = eps * industry_avg['pe']
return_pe = (target_price_pe - current_price) / current_price * 100

print(f"当前EPS: {eps:.2f} 元/股")
print(f"当前PE: {current_metrics['pe']:.2f} 倍")
print(f"行业平均PE: {industry_avg['pe']:.2f} 倍")
print(f"\n如果PE回归到行业平均:")
print(f"  目标价格 = EPS × 行业平均PE")
print(f"  目标价格 = {eps:.2f} × {industry_avg['pe']:.2f} = {target_price_pe:.2f} 元")
print(f"  预期收益率: {return_pe:+.1f}%")

# 基于行业平均PB的目标价格
bps = net_assets / total_shares  # 每股净资产
target_price_pb = bps * industry_avg['pb']
return_pb = (target_price_pb - current_price) / current_price * 100

print(f"\n当前BPS: {bps:.2f} 元/股")
print(f"当前PB: {current_metrics['pb']:.2f} 倍")
print(f"行业平均PB: {industry_avg['pb']:.2f} 倍")
print(f"\n如果PB回归到行业平均:")
print(f"  目标价格 = BPS × 行业平均PB")
print(f"  目标价格 = {bps:.2f} × {industry_avg['pb']:.2f} = {target_price_pb:.2f} 元")
print(f"  预期收益率: {return_pb:+.1f}%")

# 综合估值（PE和PB的平均）
target_price_avg = (target_price_pe + target_price_pb) / 2
return_avg = (target_price_avg - current_price) / current_price * 100

print(f"\n综合估值目标价格: {target_price_avg:.2f} 元")
print(f"预期收益率: {return_avg:+.1f}%")

## 8. 估值情景总结

In [ ]:
# 创建情景对比表
scenarios_data = []

# 情景1: 当前估值
scenarios_data.append(['当前估值', current_metrics['pe'], current_metrics['pb'], current_metrics['ps'], current_price, 0])

# 情景2: PE回归到行业平均
scenarios_data.append(['PE→行业平均', industry_avg['pe'], current_metrics['pb'], current_metrics['ps'], target_price_pe, return_pe])

# 情景3: PB回归到行业平均
scenarios_data.append(['PB→行业平均', current_metrics['pe'], industry_avg['pb'], current_metrics['ps'], target_price_pb, return_pb])

# 情景4: 全面回归行业平均
scenarios_data.append(['全面回归', industry_avg['pe'], industry_avg['pb'], industry_avg['ps'], target_price_avg, return_avg])

# 情景5: 行业最低估值（悲观）
min_pe = peer_companies_data['pe'].min()
min_pb = peer_companies_data['pb'].min()
target_price_pessimistic = (eps * min_pe + bps * min_pb) / 2
return_pessimistic = (target_price_pessimistic - current_price) / current_price * 100
scenarios_data.append(['行业最低估值', min_pe, min_pb, industry_avg['ps'], target_price_pessimistic, return_pessimistic])

# 情景6: 行业最高估值（乐观）
max_pe = peer_companies_data['pe'].max()
max_pb = peer_companies_data['pb'].max()
target_price_optimistic = (eps * max_pe + bps * max_pb) / 2
return_optimistic = (target_price_optimistic - current_price) / current_price * 100
scenarios_data.append(['行业最高估值', max_pe, max_pb, industry_avg['ps'], target_price_optimistic, return_optimistic])

df_scenarios = pd.DataFrame(scenarios_data, columns=['情景', 'PE', 'PB', 'PS', '目标价格(元)', '预期收益率(%)'])

print('\n=== 估值情景对比表 ===')
print(df_scenarios.to_string(index=False))

# 可视化情景分析
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 左图：目标价格对比
colors_scenario = ['gray', 'blue', 'green', 'orange', 'red', 'green']
bars1 = ax1.barh(df_scenarios['情景'], df_scenarios['目标价格(元)'], color=colors_scenario, alpha=0.7)
ax1.axvline(x=current_price, color='black', linestyle='--', label='当前价格')
ax1.set_xlabel('目标价格 (元)', fontproperties=font_prop, fontsize=12)
ax1.set_title('不同估值情景下的目标价格', fontproperties=font_prop, fontsize=13, fontweight='bold')
ax1.legend(prop=font_prop)
for label in ax1.get_xticklabels():
    label.set_fontproperties(font_prop)
for label in ax1.get_yticklabels():
    label.set_fontproperties(font_prop)
ax1.grid(True, alpha=0.3, axis='x')

# 右图：预期收益率对比
colors_return = ['green' if r > 0 else 'red' for r in df_scenarios['预期收益率(%)']]
bars2 = ax2.barh(df_scenarios['情景'], df_scenarios['预期收益率(%)'], color=colors_return, alpha=0.7)
ax2.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax2.set_xlabel('预期收益率 (%)', fontproperties=font_prop, fontsize=12)
ax2.set_title('不同估值情景下的预期收益率', fontproperties=font_prop, fontsize=13, fontweight='bold')
for label in ax2.get_xticklabels():
    label.set_fontproperties(font_prop)
for label in ax2.get_yticklabels():
    label.set_fontproperties(font_prop)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 9. 相对估值结论

In [ ]:
print('\n' + '='*60)
print('相对估值分析结论')
print('='*60)

print(f"\n1. 当前估值水平:")
if current_metrics['pe'] < industry_avg['pe']:
    print(f"   PE ({current_metrics['pe']:.1f}倍) 低于行业平均 ({industry_avg['pe']:.1f}倍)")
    print(f"   → 股价可能被低估")
else:
    print(f"   PE ({current_metrics['pe']:.1f}倍) 高于行业平均 ({industry_avg['pe']:.1f}倍)")
    print(f"   → 股价可能被高估")

print(f"\n2. 估值回归空间:")
print(f"   如果估值回归到行业平均，预期收益率: {return_avg:+.1f}%")

if abs(return_avg) < 10:
    risk_level = "低风险 - 估值接近行业均值"
elif abs(return_avg) < 30:
    risk_level = "中等风险 - 估值有一定偏离"
else:
    risk_level = "高风险 - 估值偏离较大"

print(f"\n3. 投资建议:")
print(f"   {risk_level}")

if return_avg > 10:
    print(f"   建议: 估值偏低，具有投资价值")
elif return_avg < -10:
    print(f"   建议: 估值偏高，需谨慎投资")
else:
    print(f"   建议: 估值合理，可正常投资")

print(f"\n4. 敏感性分析:")
print(f"   - 悡价每变动1元，对应收益率变动: {100/current_price:.1f}%")
print(f"   - PE每变动1倍，对应目标价格变动: {eps:.2f} 元")